# Fretboard Assignment: Original vs Playability vs Combined-All Comparison

This notebook extends the fretboard assignment prototype with a stronger **playability-aware** layer.

It includes:

- Standard guitar fretboard layout and MIDI lookup tables
- 24-key scale data and diatonic chord knowledge
- Chord-tone definitions and simple chord recognition
- GuitarSet `.jams` parsing
- Baseline tab assignment methods
- Viterbi-style sequence optimization
- Playability rules: comfortable span, reachable span, hand position, open-string preferences, awkward fingering penalties, chord feasibility constraints
- Full evaluation outputs saved to CSV

**Colab behavior in this version:**

1. Mounts Google Drive at `/content/drive`
2. Reads GuitarSet data from a detected `FullGuitarSetData` folder
3. Saves CSV outputs directly to:

```text
/content/drive/MyDrive/Capstone/outputs/fretboard_playability/
```

So in Google Drive, the CSVs should appear under:

```text
My Drive / Capstone / outputs / fretboard_playability
```


In [1]:
from pathlib import Path
from itertools import product
from collections import defaultdict
import json
import math
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)


In [2]:
# Mount Google Drive when running in Colab.
# This must run before any /content/drive/MyDrive/... paths are used.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted successfully.')
except ModuleNotFoundError:
    print('Not running in Colab. Skipping Google Drive mount.')


Mounted at /content/drive
Google Drive mounted successfully.


## 1. Configuration

This notebook assumes the GuitarSet data is somewhere in your mounted Google Drive, ideally one of these:

```text
/content/drive/MyDrive/Capstone/FullGuitarSetData
/content/drive/MyDrive/FullGuitarSetData
```

Expected data structure:

```text
FullGuitarSetData/
├── JamsFiles/
└── AudioFiles/
```

The CSV outputs are saved to:

```text
/content/drive/MyDrive/Capstone/outputs/fretboard_playability/
```

If the notebook cannot find the data folder automatically, update `DATA_ROOT_CANDIDATES` in the next cell.


In [3]:
# -------------------------
# USER CONFIG
# -------------------------

# Where outputs should go in Google Drive.
# This creates: My Drive / Capstone / outputs / fretboard_playability
# In local/non-Colab execution, this path may be created locally, but in Colab it writes to Drive after mounting.
CAPSTONE_ROOT = Path('/content/drive/MyDrive/Capstone')
OUTPUT_DIR = CAPSTONE_ROOT / 'outputs' / 'fretboard_playability'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Candidate locations for the GuitarSet data folder.
# Update/add to this list if your FullGuitarSetData or GuitarSet folder is somewhere else.
# The notebook will choose the first candidate that actually contains .jams files.
DATA_ROOT_CANDIDATES = [
    Path('/content/drive/MyDrive/Capstone/FullGuitarSetData'),
    Path('/content/drive/MyDrive/FullGuitarSetData'),
    Path('/content/drive/MyDrive/Capstone/GuitarSet'),
    Path('/content/drive/MyDrive/GuitarSet'),
    Path('/content/drive/MyDrive/Capstone'),
    Path('/content/drive/MyDrive'),
    Path('/mnt/data/fretwork_repo/GuitarSet'),
    Path('/mnt/data/fretwork_repo'),
]

MAX_FRET = 24
OPEN_STRING_MIDI = [40, 45, 50, 55, 59, 64]  # E2, A2, D3, G3, B3, E4
STRING_NAMES = ['low_E', 'A', 'D', 'G', 'B', 'high_E']
ONSET_TOLERANCE_SECONDS = 0.035

COMFORTABLE_SPAN = 5
MAX_REACHABLE_SPAN = 7
LARGE_JUMP_THRESHOLD = 5
MAX_GROUP_CANDIDATES = 25  # optimization cap for chord candidate combinations, not a demo note limit

print(f'OUTPUT_DIR: {OUTPUT_DIR.resolve()}')
print('OUTPUT_DIR exists:', OUTPUT_DIR.exists())
print('\nData root candidates visible to this runtime:')
for p in DATA_ROOT_CANDIDATES:
    print(f' - {p} | exists: {p.exists()}')

if Path('/content/drive/MyDrive').exists():
    print('\nTop-level MyDrive folders/files visible to Colab:')
    for p in list(Path('/content/drive/MyDrive').iterdir())[:25]:
        print(' -', p.name)


OUTPUT_DIR: /content/drive/.shortcut-targets-by-id/1JNqe8bukG93wCWVxbk7SKlNvVyIZHyTC/Capstone/outputs/fretboard_playability
OUTPUT_DIR exists: True

Data root candidates visible to this runtime:
 - /content/drive/MyDrive/Capstone/FullGuitarSetData | exists: True
 - /content/drive/MyDrive/FullGuitarSetData | exists: False
 - /content/drive/MyDrive/Capstone/GuitarSet | exists: True
 - /content/drive/MyDrive/GuitarSet | exists: False
 - /content/drive/MyDrive/Capstone | exists: True
 - /content/drive/MyDrive | exists: True
 - /mnt/data/fretwork_repo/GuitarSet | exists: False
 - /mnt/data/fretwork_repo | exists: False

Top-level MyDrive folders/files visible to Colab:
 - DATASCI 201 Belmont Report Assignment.gdoc
 - Critique a Design Pt 3.gdoc
 - Unit 08 Assignment.gdoc
 - The Revival of High School Shop Classes: A Hands-On Comeback Story.gslides
 - Unit 9 Assignment - Aaron Luong.mp4
 - Unit 9 Assignment - Slides & Notes.gdoc
 - Career Compass.gdoc
 - Career Compass.mp4
 - Career Compass 

## 2. Fretboard Layout and MIDI Lookups

In [4]:
def build_fretboard(open_string_midi=OPEN_STRING_MIDI, max_fret=MAX_FRET):
    rows = []
    for string_idx, open_midi in enumerate(open_string_midi):
        for fret in range(max_fret + 1):
            midi = open_midi + fret
            rows.append({
                'string': string_idx,
                'string_name': STRING_NAMES[string_idx],
                'fret': fret,
                'midi': midi,
                'pitch_class': midi % 12,
            })
    return pd.DataFrame(rows)

fretboard_df = build_fretboard()
MIDI_TO_POSITIONS = defaultdict(list)
for row in fretboard_df.to_dict('records'):
    MIDI_TO_POSITIONS[int(row['midi'])].append({
        'string': int(row['string']),
        'string_name': row['string_name'],
        'fret': int(row['fret']),
        'midi': int(row['midi']),
        'pitch_class': int(row['pitch_class']),
    })

def get_possible_positions(midi_note, max_fret=MAX_FRET):
    midi_note = int(round(midi_note))
    return [p for p in MIDI_TO_POSITIONS.get(midi_note, []) if 0 <= p['fret'] <= max_fret]

print('Fretboard rows:', len(fretboard_df))
display(fretboard_df.head(12))
print('Example positions for MIDI 64 / E4:')
display(pd.DataFrame(get_possible_positions(64)))


Fretboard rows: 150


,string,string_name,fret,midi,pitch_class
0,0,low_E,0,40,4
1,0,low_E,1,41,5
2,0,low_E,2,42,6
3,0,low_E,3,43,7
4,0,low_E,4,44,8
5,0,low_E,5,45,9
6,0,low_E,6,46,10
7,0,low_E,7,47,11
8,0,low_E,8,48,0
9,0,low_E,9,49,1


Example positions for MIDI 64 / E4:


,string,string_name,fret,midi,pitch_class
0,0,low_E,24,64,4
1,1,A,19,64,4
2,2,D,14,64,4
3,3,G,9,64,4
4,4,B,5,64,4
5,5,high_E,0,64,4


## 3. Scale, Key, and Diatonic Chord Knowledge

In [5]:
PITCH_CLASS_NAMES_SHARP = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
NOTE_TO_PC = {name: i for i, name in enumerate(PITCH_CLASS_NAMES_SHARP)}
NOTE_TO_PC.update({'Db': 1, 'Eb': 3, 'Gb': 6, 'Ab': 8, 'Bb': 10})
PC_TO_NOTE = {i: name for i, name in enumerate(PITCH_CLASS_NAMES_SHARP)}

MAJOR_STEPS = [2, 2, 1, 2, 2, 2, 1]
MINOR_STEPS = [2, 1, 2, 2, 1, 2, 2]
MAJOR_QUALITIES = ['maj', 'min', 'min', 'maj', 'maj', 'min', 'dim']
MINOR_QUALITIES = ['min', 'dim', 'maj', 'min', 'min', 'maj', 'maj']

def derive_scale(root_pc, mode='major'):
    steps = MAJOR_STEPS if mode == 'major' else MINOR_STEPS
    pcs = [root_pc]
    cur = root_pc
    for step in steps[:-1]:
        cur = (cur + step) % 12
        pcs.append(cur)
    return pcs

def build_key_database():
    rows = []
    for root_name, root_pc in NOTE_TO_PC.items():
        if 'b' in root_name:
            continue
        for mode in ['major', 'minor']:
            scale_pcs = derive_scale(root_pc, mode)
            qualities = MAJOR_QUALITIES if mode == 'major' else MINOR_QUALITIES
            chords = []
            for degree, (pc, qual) in enumerate(zip(scale_pcs, qualities), start=1):
                chords.append({
                    'degree': degree,
                    'root_pc': pc,
                    'root': PC_TO_NOTE[pc],
                    'quality': qual,
                    'symbol': f'{PC_TO_NOTE[pc]}:{qual}',
                })
            rows.append({
                'key': f'{root_name} {mode}',
                'root': root_name,
                'root_pc': root_pc,
                'mode': mode,
                'scale_pcs': scale_pcs,
                'scale_notes': [PC_TO_NOTE[pc] for pc in scale_pcs],
                'diatonic_chords': chords,
            })
    return pd.DataFrame(rows)

key_db = build_key_database()
display(key_db.head())
print('D major diatonic chords:')
d_major = key_db[key_db['key'] == 'D major'].iloc[0]
print([c['symbol'] for c in d_major['diatonic_chords']])


,key,root,root_pc,mode,scale_pcs,scale_notes,diatonic_chords
0,C major,C,0,major,"[0, 2, 4, 5, 7, 9, 11]","[C, D, E, F, G, A, B]","[{'degree': 1, 'root_pc': 0, 'root': 'C', 'qua..."
1,C minor,C,0,minor,"[0, 2, 3, 5, 7, 8, 10]","[C, D, D#, F, G, G#, A#]","[{'degree': 1, 'root_pc': 0, 'root': 'C', 'qua..."
2,C# major,C#,1,major,"[1, 3, 5, 6, 8, 10, 0]","[C#, D#, F, F#, G#, A#, C]","[{'degree': 1, 'root_pc': 1, 'root': 'C#', 'qu..."
3,C# minor,C#,1,minor,"[1, 3, 4, 6, 8, 9, 11]","[C#, D#, E, F#, G#, A, B]","[{'degree': 1, 'root_pc': 1, 'root': 'C#', 'qu..."
4,D major,D,2,major,"[2, 4, 6, 7, 9, 11, 1]","[D, E, F#, G, A, B, C#]","[{'degree': 1, 'root_pc': 2, 'root': 'D', 'qua..."


D major diatonic chords:
['D:maj', 'E:min', 'F#:min', 'G:maj', 'A:maj', 'B:min', 'C#:dim']


## 4. Chord Knowledge and Recognition

In [6]:
CHORD_INTERVALS = {
    'maj': [0, 4, 7],
    'min': [0, 3, 7],
    'dim': [0, 3, 6],
    'aug': [0, 4, 8],
    '7': [0, 4, 7, 10],
    'maj7': [0, 4, 7, 11],
    'min7': [0, 3, 7, 10],
    'm7': [0, 3, 7, 10],
    'sus4': [0, 5, 7],
    'sus2': [0, 2, 7],
    '5': [0, 7],
}
QUALITY_ALIASES = {'M': 'maj', 'major': 'maj', '': 'maj', 'm': 'min', 'minor': 'min', 'dom7': '7'}

def normalize_quality(q):
    if q is None:
        return 'maj'
    q = str(q).strip()
    return QUALITY_ALIASES.get(q, q)

def chord_tones(root_pc, quality='maj'):
    quality = normalize_quality(quality)
    intervals = CHORD_INTERVALS.get(quality, CHORD_INTERVALS['maj'])
    return sorted({(root_pc + i) % 12 for i in intervals})

def parse_chord_symbol(symbol):
    if symbol is None:
        return None
    s = str(symbol).strip()
    # Remove inversion/bass-note suffixes such as D:7/1 or C:maj/G before parsing quality.
    s = s.split('/')[0]
    if s in ['N', 'X', 'nan', 'None', '']:
        return None
    if ':' in s:
        root, qual = s.split(':', 1)
    else:
        m = re.match(r'^([A-G](?:#|b)?)(.*)$', s)
        if not m:
            return None
        root, qual = m.group(1), m.group(2)
    if root not in NOTE_TO_PC:
        return None
    qual = normalize_quality(qual)
    return {'root': root, 'root_pc': NOTE_TO_PC[root], 'quality': qual, 'tones': chord_tones(NOTE_TO_PC[root], qual)}

def recognize_chord_from_pitches(midi_pitches, allowed_qualities=('maj', 'min', 'dim', '7', 'maj7', 'min7')):
    pcs = sorted({int(round(m)) % 12 for m in midi_pitches})
    if not pcs:
        return None
    best = None
    for root_pc in range(12):
        for qual in allowed_qualities:
            tones = set(chord_tones(root_pc, qual))
            pcs_set = set(pcs)
            precision = len(pcs_set & tones) / max(len(pcs_set), 1)
            recall = len(pcs_set & tones) / max(len(tones), 1)
            score = 2 * precision * recall / (precision + recall + 1e-9)
            cand = {'symbol': f'{PC_TO_NOTE[root_pc]}:{qual}', 'root_pc': root_pc, 'quality': qual, 'tones': sorted(tones), 'score': score}
            if best is None or cand['score'] > best['score']:
                best = cand
    return best

print(parse_chord_symbol('D:maj'))
print(recognize_chord_from_pitches([62, 66, 69]))


{'root': 'D', 'root_pc': 2, 'quality': 'maj', 'tones': [2, 6, 9]}
{'symbol': 'D:maj', 'root_pc': 2, 'quality': 'maj', 'tones': [2, 6, 9], 'score': 0.9999999995}


## 5. GuitarSet JAMS Parsing

This parser avoids requiring the external `jams` package. It directly reads the JSON-like `.jams` files.

In [7]:
def find_jams_dir(data_root):
    """Return a directory containing .jams files under data_root, or None if not found."""
    candidates = [
        data_root / 'JamsFiles',
        data_root / 'Annotations',
        data_root / 'GuitarSet' / 'Annotations',
        data_root / 'FullGuitarSetData' / 'JamsFiles',
        data_root / 'FullGuitarSetData' / 'Annotations',
        data_root,
    ]
    for c in candidates:
        if c.exists() and list(c.glob('*.jams')):
            return c

    # Last-resort recursive search under this candidate.
    # Limit to the first match to avoid loading the full Drive tree unnecessarily.
    if data_root.exists():
        try:
            for match in data_root.rglob('*.jams'):
                return match.parent
        except Exception as e:
            print(f'Could not recursively search {data_root}: {e}')
    return None

def choose_data_root_and_jams_dir(candidates):
    checked = []
    for root in candidates:
        checked.append((root, root.exists()))
        if not root.exists():
            continue
        jams_dir = find_jams_dir(root)
        if jams_dir is not None:
            return root, jams_dir

    print('Could not find .jams files automatically.')
    print('Checked these DATA_ROOT_CANDIDATES:')
    for root, exists in checked:
        print(f' - {root} | exists: {exists}')
    raise FileNotFoundError(
        'Could not find any .jams files. Add the correct GuitarSet/FullGuitarSetData path to DATA_ROOT_CANDIDATES.'
    )

DATA_ROOT, JAMS_DIR = choose_data_root_and_jams_dir(DATA_ROOT_CANDIDATES)
JAMS_FILES = sorted(JAMS_DIR.glob('*.jams'))
print(f'DATA_ROOT selected: {DATA_ROOT}')
print(f'Found {len(JAMS_FILES)} JAMS files in {JAMS_DIR}')
print('\n'.join(p.name for p in JAMS_FILES[:10]))

def get_annotation_data(annotation):
    data = annotation.get('data', [])
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        keys = ['time', 'duration', 'value', 'confidence']
        n = len(data.get('time', []))
        return [{k: data.get(k, [None] * n)[i] for k in keys} for i in range(n)]
    return []

def parse_string_from_data_source(data_source):
    """GuitarSet stores each string as a separate note_midi annotation with data_source 0-5."""
    try:
        s = int(data_source)
        return s if 0 <= s <= 5 else None
    except Exception:
        return None

def parse_jams_file(path):
    with open(path, 'r') as f:
        jam = json.load(f)
    notes, chords, beats = [], [], []
    tempo, key = None, None
    for ann in jam.get('annotations', []):
        ns = ann.get('namespace')
        rows = get_annotation_data(ann)
        data_source = ann.get('annotation_metadata', {}).get('data_source', '')
        if ns == 'note_midi':
            inferred_string = parse_string_from_data_source(data_source)
            for r in rows:
                v = r.get('value')
                if isinstance(v, dict):
                    midi = v.get('midi_note') or v.get('note') or v.get('pitch')
                    string = v.get('string', inferred_string)
                    fret = v.get('fret')
                else:
                    midi = v
                    string = inferred_string
                    fret = None
                if midi is None:
                    continue
                midi_int = int(round(float(midi)))
                # GuitarSet note_midi annotations usually give string via annotation data_source.
                # If fret is not explicitly stored, derive it from MIDI pitch and the open string pitch.
                if string is not None and fret is None:
                    fret = midi_int - OPEN_STRING_MIDI[int(string)]
                notes.append({
                    'start': float(r.get('time', 0.0)),
                    'duration': float(r.get('duration', 0.0) or 0.0),
                    'midi': midi_int,
                    'pitch_class': midi_int % 12,
                    'true_string': None if string is None else int(string),
                    'true_fret': None if fret is None else int(round(float(fret))),
                    'source': data_source,
                })
        elif ns in ['chord', 'chord_harte']:
            for r in rows:
                start = float(r.get('time', 0.0))
                duration = float(r.get('duration', 0.0) or 0.0)
                chord_label = r.get('value')
                chords.append({
                    'start': start,
                    'duration': duration,
                    'end': start + duration,
                    'chord': chord_label,
                    'parsed': parse_chord_symbol(chord_label),
                })
        elif ns in ['beat', 'beat_position']:
            for r in rows:
                beats.append(float(r.get('time', 0.0)))
        elif ns == 'key_mode':
            if rows:
                key = rows[0].get('value')
        elif ns == 'tempo':
            if rows:
                tempo = rows[0].get('value')
    notes = sorted(notes, key=lambda x: (x['start'], x['midi']))
    chords = sorted(chords, key=lambda x: x['start'])
    return {'recording': path.stem, 'path': str(path), 'notes': notes, 'chords': chords, 'beats': beats, 'tempo': tempo, 'key': key}

records = [parse_jams_file(p) for p in JAMS_FILES]
print('Parsed records:', len(records))
if records:
    print('Example record:', records[0]['recording'])
    print('Notes:', len(records[0]['notes']), 'Chords:', len(records[0]['chords']), 'Key:', records[0]['key'])
    display(pd.DataFrame(records[0]['notes']).head())
else:
    raise ValueError('No records parsed. Check JAMS_FILES and DATA_ROOT_CANDIDATES.')


DATA_ROOT selected: /content/drive/MyDrive/Capstone/FullGuitarSetData
Found 360 JAMS files in /content/drive/MyDrive/Capstone/FullGuitarSetData/JamsFiles
00_BN1-129-Eb_comp.jams
00_BN1-129-Eb_solo.jams
00_BN1-147-Gb_comp.jams
00_BN1-147-Gb_solo.jams
00_BN2-131-B_comp.jams
00_BN2-131-B_solo.jams
00_BN2-166-Ab_comp.jams
00_BN2-166-Ab_solo.jams
00_BN3-119-G_comp.jams
00_BN3-119-G_solo.jams
Parsed records: 360
Example record: 00_BN1-129-Eb_comp
Notes: 133 Chords: 12 Key: Eb:major


,start,duration,midi,pitch_class,true_string,true_fret,source
0,0.048816,0.423764,51,3,1,6,1
1,0.049791,0.452789,65,5,4,6,4
2,0.052717,0.458594,62,2,3,7,3
3,0.519995,0.417959,51,3,1,6,1
4,0.722036,0.859138,58,10,2,8,2


## 6. Context Helpers: Key and Chord at Each Note

In [8]:
def infer_key_from_filename(recording_name):
    parts = recording_name.split('_')
    if len(parts) >= 2:
        middle = parts[1]
        key_guess = middle.split('-')[-1]
        if key_guess in NOTE_TO_PC:
            return f'{key_guess} major'
    return None

def get_key_info(key_label):
    if key_label is None:
        return None
    s = str(key_label).replace(':', ' ').strip()
    # Normalize GuitarSet-style labels such as D:major into D major.
    toks = s.split()
    if len(toks) == 1 and toks[0] in NOTE_TO_PC:
        s = f'{toks[0]} major'
    match = key_db[key_db['key'] == s]
    return match.iloc[0].to_dict() if len(match) else None

def chord_end_time(c):
    # Some parsed chord dictionaries have duration but not an explicit end time.
    # This helper keeps the rest of the notebook robust either way.
    start = float(c.get('start', 0.0))
    if c.get('end') is not None:
        return float(c['end'])
    return start + float(c.get('duration', 0.0) or 0.0)

def chord_at_time(chords, t):
    for c in chords:
        start = float(c.get('start', 0.0))
        end = chord_end_time(c)
        if start <= t < end:
            return c
    return None

def enrich_notes_with_context(record):
    key_label = record.get('key') or infer_key_from_filename(record['recording'])
    key_info = get_key_info(key_label)
    out = []
    for n in record['notes']:
        c = chord_at_time(record['chords'], n['start'])
        row = dict(n)
        row['key_label'] = key_label
        row['in_key'] = None if key_info is None else (n['pitch_class'] in set(key_info['scale_pcs']))
        row['chord_label'] = None if c is None else c['chord']
        parsed_chord = None if c is None else c.get('parsed') or parse_chord_symbol(c.get('chord'))
        row['in_chord'] = None if parsed_chord is None else (n['pitch_class'] in set(parsed_chord['tones']))
        out.append(row)
    return out

sample_context = pd.DataFrame(enrich_notes_with_context(records[0]))
display(sample_context.head())


,start,duration,midi,pitch_class,true_string,true_fret,source,key_label,in_key,chord_label,in_chord
0,0.048816,0.423764,51,3,1,6,1,Eb:major,None,D#:maj,True
1,0.049791,0.452789,65,5,4,6,4,Eb:major,None,D#:maj,False
2,0.052717,0.458594,62,2,3,7,3,Eb:major,None,D#:maj,False
3,0.519995,0.417959,51,3,1,6,1,Eb:major,None,D#:maj,True
4,0.722036,0.859138,58,10,2,8,2,Eb:major,None,D#:maj,True


## 7. Playability Rules and Scoring

In [9]:
def estimate_hand_position_from_frets(frets):
    fretted = [f for f in frets if f > 0]
    return 0 if not fretted else int(round(np.median(fretted)))

def group_span(frets):
    fretted = [f for f in frets if f > 0]
    return 0 if len(fretted) <= 1 else max(fretted) - min(fretted)

def awkward_fingering_penalty(position, hand_center):
    fret = position['fret']
    if fret == 0:
        return 0.0
    distance = abs(fret - hand_center)
    if distance <= 2:
        return 0.0
    if distance <= COMFORTABLE_SPAN:
        return 0.5 * (distance - 2)
    if distance <= MAX_REACHABLE_SPAN:
        return 2.0 + (distance - COMFORTABLE_SPAN)
    return 10.0 + 2.0 * (distance - MAX_REACHABLE_SPAN)

def group_playability_cost(group_positions):
    if not group_positions:
        return 0.0
    strings = [p['string'] for p in group_positions]
    frets = [p['fret'] for p in group_positions]
    fretted = [f for f in frets if f > 0]
    cost = 0.0
    if len(strings) != len(set(strings)):
        return float('inf')
    span = group_span(frets)
    if span > COMFORTABLE_SPAN:
        cost += 2.0 * (span - COMFORTABLE_SPAN)
    if span > MAX_REACHABLE_SPAN:
        cost += 25.0 * (span - MAX_REACHABLE_SPAN)
    if fretted and min(fretted) <= 2 and max(fretted) >= 9:
        cost += 8.0
    if len(strings) >= 2:
        string_span = max(strings) - min(strings)
        if string_span > 4 and len(strings) <= 3:
            cost += 1.5 * (string_span - 4)
    hand_center = estimate_hand_position_from_frets(frets)
    cost += sum(awkward_fingering_penalty(p, hand_center) for p in group_positions)
    if any(f == 0 for f in frets) and fretted and max(fretted) > 7:
        cost += 3.0
    return cost

def transition_cost(prev_group, curr_group):
    if prev_group is None or curr_group is None:
        return 0.0
    prev_frets = [p['fret'] for p in prev_group]
    curr_frets = [p['fret'] for p in curr_group]
    prev_strings = [p['string'] for p in prev_group]
    curr_strings = [p['string'] for p in curr_group]
    prev_center = estimate_hand_position_from_frets(prev_frets)
    curr_center = estimate_hand_position_from_frets(curr_frets)
    cost = 1.2 * abs(curr_center - prev_center) + 0.25 * abs(np.mean(curr_strings) - np.mean(prev_strings))
    if len(prev_group) == 1 and len(curr_group) == 1:
        pf, cf = prev_group[0]['fret'], curr_group[0]['fret']
        ps, cs = prev_group[0]['string'], curr_group[0]['string']
        cost += 0.8 * abs(cf - pf) + 0.35 * abs(cs - ps)
        if abs(cf - pf) > LARGE_JUMP_THRESHOLD:
            cost += 4.0 + abs(cf - pf) - LARGE_JUMP_THRESHOLD
        if cf == 0 and pf > 7:
            cost += 2.0
    return cost

def context_cost(group_notes, group_positions):
    cost = 0.0
    for n, p in zip(group_notes, group_positions):
        if n.get('in_chord') is False:
            cost += 0.15
        if n.get('in_key') is False:
            cost += 0.10
    return cost


## 8. Group Notes by Onset

In [10]:
def group_notes_by_onset(notes, tolerance=ONSET_TOLERANCE_SECONDS):
    if not notes:
        return []
    notes_sorted = sorted(notes, key=lambda x: (x['start'], x.get('true_string') if x.get('true_string') is not None else 99, x['midi']))
    groups, current = [], [notes_sorted[0]]
    group_start = notes_sorted[0]['start']
    for n in notes_sorted[1:]:
        if abs(n['start'] - group_start) <= tolerance:
            current.append(n)
        else:
            groups.append(current)
            current = [n]
            group_start = n['start']
    groups.append(current)
    return groups

def enrich_candidate(candidate):
    positions = candidate['positions']
    frets = [p['fret'] for p in positions]
    strings = [p['string'] for p in positions]
    candidate['center'] = estimate_hand_position_from_frets(frets)
    candidate['avg_string'] = float(np.mean(strings)) if strings else 0.0
    candidate['is_single'] = len(positions) == 1
    candidate['single_fret'] = positions[0]['fret'] if len(positions) == 1 else np.nan
    candidate['single_string'] = positions[0]['string'] if len(positions) == 1 else np.nan
    return candidate

def candidate_groups_for_notes(group_notes, max_candidates=MAX_GROUP_CANDIDATES):
    position_lists = []
    for n in group_notes:
        pos = get_possible_positions(n['midi'])
        if not pos:
            return []
        position_lists.append(pos)
    candidates = []
    for combo in product(*position_lists):
        combo = list(combo)
        if len(combo) > 1 and len({p['string'] for p in combo}) != len(combo):
            continue
        base_cost = group_playability_cost(combo) + context_cost(group_notes, combo)
        if math.isfinite(base_cost):
            candidates.append(enrich_candidate({'positions': combo, 'base_cost': base_cost}))
    if not candidates:
        for combo in product(*position_lists):
            combo = list(combo)
            base_cost = group_playability_cost(combo)
            if math.isinf(base_cost):
                base_cost = 1000.0
            candidates.append(enrich_candidate({'positions': combo, 'base_cost': base_cost}))
    return sorted(candidates, key=lambda c: c['base_cost'])[:max_candidates]

sample_groups = group_notes_by_onset(enrich_notes_with_context(records[0]))
print('Number of onset groups:', len(sample_groups))
print('First group size:', len(sample_groups[0]))
print('First group candidates:', len(candidate_groups_for_notes(sample_groups[0])))


Number of onset groups: 76
First group size: 3
First group candidates: 25


## 9. Baseline Assignment Methods

In [11]:
def choose_lowest_fret(midi):
    pos = get_possible_positions(midi)
    return None if not pos else min(pos, key=lambda p: (p['fret'], p['string']))

def choose_highest_string(midi):
    pos = get_possible_positions(midi)
    return None if not pos else max(pos, key=lambda p: (p['string'], -p['fret']))

def assign_baseline_lowest_fret(notes):
    out = []
    for n in notes:
        p = choose_lowest_fret(n['midi'])
        row = dict(n)
        row.update({'pred_string': None if p is None else p['string'], 'pred_fret': None if p is None else p['fret'], 'method': 'lowest_fret'})
        out.append(row)
    return out

def assign_baseline_highest_string(notes):
    out = []
    for n in notes:
        p = choose_highest_string(n['midi'])
        row = dict(n)
        row.update({'pred_string': None if p is None else p['string'], 'pred_fret': None if p is None else p['fret'], 'method': 'highest_string'})
        out.append(row)
    return out

def assign_nearest_previous(notes):
    groups = group_notes_by_onset(notes)
    pred_rows, prev_group = [], None
    for g in groups:
        candidates = candidate_groups_for_notes(g)
        if not candidates:
            continue
        best = min(candidates, key=lambda c: c['base_cost'] + transition_cost(prev_group, c['positions']))
        prev_group = best['positions']
        for n, p in zip(g, best['positions']):
            row = dict(n)
            row.update({'pred_string': p['string'], 'pred_fret': p['fret'], 'method': 'nearest_previous'})
            pred_rows.append(row)
    return sorted(pred_rows, key=lambda x: (x['start'], x.get('true_string') if x.get('true_string') is not None else 99, x['midi']))


## 10. Viterbi-Style Playability Assignment

In [12]:
def transition_cost_matrix(prev_cands, curr_cands):
    prev_center = np.array([c['center'] for c in prev_cands], dtype=float)
    curr_center = np.array([c['center'] for c in curr_cands], dtype=float)
    prev_str = np.array([c['avg_string'] for c in prev_cands], dtype=float)
    curr_str = np.array([c['avg_string'] for c in curr_cands], dtype=float)
    mat = 1.2 * np.abs(prev_center[:, None] - curr_center[None, :])
    mat += 0.25 * np.abs(prev_str[:, None] - curr_str[None, :])

    prev_single = np.array([c['is_single'] for c in prev_cands], dtype=bool)
    curr_single = np.array([c['is_single'] for c in curr_cands], dtype=bool)
    single_mask = prev_single[:, None] & curr_single[None, :]
    if single_mask.any():
        pf = np.array([c['single_fret'] for c in prev_cands], dtype=float)[:, None]
        cf = np.array([c['single_fret'] for c in curr_cands], dtype=float)[None, :]
        ps = np.array([c['single_string'] for c in prev_cands], dtype=float)[:, None]
        cs = np.array([c['single_string'] for c in curr_cands], dtype=float)[None, :]
        fret_diff = np.abs(cf - pf)
        string_diff = np.abs(cs - ps)
        extra = 0.8 * fret_diff + 0.35 * string_diff
        extra += np.where(fret_diff > LARGE_JUMP_THRESHOLD, 4.0 + fret_diff - LARGE_JUMP_THRESHOLD, 0.0)
        extra += np.where((cf == 0) & (pf > 7), 2.0, 0.0)
        mat += np.where(single_mask, extra, 0.0)
    return mat

def assign_viterbi_playability(notes):
    groups = group_notes_by_onset(notes)
    all_candidates = [candidate_groups_for_notes(g) for g in groups]
    if any(len(cands) == 0 for cands in all_candidates):
        raise ValueError('At least one group has no valid candidates.')

    dp = [np.array([c['base_cost'] for c in all_candidates[0]], dtype=float)]
    backptr = [np.full(len(dp[0]), -1, dtype=int)]

    for i in range(1, len(groups)):
        prev_cands, curr_cands = all_candidates[i - 1], all_candidates[i]
        trans = transition_cost_matrix(prev_cands, curr_cands)
        curr_base = np.array([c['base_cost'] for c in curr_cands], dtype=float)
        scores = dp[i - 1][:, None] + trans + curr_base[None, :]
        curr_back = np.argmin(scores, axis=0).astype(int)
        curr_costs = scores[curr_back, np.arange(scores.shape[1])]
        dp.append(curr_costs)
        backptr.append(curr_back)

    idx = int(np.argmin(dp[-1]))
    chosen_indices = [idx]
    for i in range(len(groups) - 1, 0, -1):
        idx = int(backptr[i][idx])
        chosen_indices.append(idx)
    chosen_indices = list(reversed(chosen_indices))

    pred_rows = []
    for g, cands, ci in zip(groups, all_candidates, chosen_indices):
        positions = cands[ci]['positions']
        for n, p in zip(g, positions):
            row = dict(n)
            row.update({'pred_string': p['string'], 'pred_fret': p['fret'], 'method': 'viterbi_playability'})
            pred_rows.append(row)
    return sorted(pred_rows, key=lambda x: (x['start'], x.get('true_string') if x.get('true_string') is not None else 99, x['midi']))


## 10b. Original Teammate Algorithm + Combined-All Method

This section adds the original teammate logic into the same GuitarSet evaluation loop.

Methods added:

- `old_music_theory_greedy`: reproduces the original greedy music-theory-aware assignment style. It scores each valid position using key alignment, chord-tone membership, open-string bonus, fret-region comfort, and continuity from the previous note.
- `combined_all`: uses the new Viterbi/global optimization framework, but adds the original music-theory score as an additional candidate cost term on top of playability, span, open-string, chord/key, and transition rules.

This allows an apples-to-apples table comparing old/simple methods, the new playability method, and a combined method across the same GuitarSet records.

In [13]:

# -----------------------------------------------------------------------------
# Original teammate algorithm adapted for this notebook's data structures
# -----------------------------------------------------------------------------

OLD_THEORY_WEIGHTS = {
    'key_alignment': 1.0,
    'chord_tone': 2.0,
    'open_string_bonus': 1.0,
    'low_position_bonus': 0.5,
    'middle_neck_bonus': 0.3,
    'position_continuity': 0.5,
    'continuity_cap': 5.0,
}


def old_position_score(midi, position, note_row=None, previous_position=None, weights=None):
    """Higher-is-better score from the original music-theory-aware prototype.

    This adapts the old notebook's `score_position()` logic to the richer rows in this
    notebook. The score uses key/chord flags already computed by `enrich_notes_with_context`.
    """
    if weights is None:
        weights = OLD_THEORY_WEIGHTS

    fret = position['fret']
    score = 0.0

    if note_row is not None and note_row.get('in_key') is True:
        score += weights['key_alignment']

    if note_row is not None and note_row.get('in_chord') is True:
        score += weights['chord_tone']

    if fret == 0:
        score += weights['open_string_bonus']
    elif fret <= 3:
        score += weights['low_position_bonus']
    elif 4 <= fret <= 12:
        score += weights['middle_neck_bonus']

    if previous_position is not None:
        prev_fret = previous_position['fret']
        if prev_fret > 0 and fret > 0:
            fret_distance = min(abs(fret - prev_fret), weights['continuity_cap'])
            score -= weights['position_continuity'] * (fret_distance ** 0.5)

    return float(score)


def assign_old_music_theory_greedy(notes):
    """Original teammate music-theory-aware assignment, evaluated over GuitarSet.

    Greedy per-note method:
    - enumerate valid positions for each MIDI note
    - score each position using old key/chord/comfort/continuity rules
    - choose the best local position

    For simultaneous notes, this remains per-note and can therefore reveal duplicate-string
    violations, which is useful when comparing old vs. new playability rules.
    """
    pred_rows = []
    previous_position = None

    for n in sorted(notes, key=lambda x: (x['start'], x.get('true_string') if x.get('true_string') is not None else 99, x['midi'])):
        positions = get_possible_positions(n['midi'])
        row = dict(n)

        if not positions:
            row.update({'pred_string': None, 'pred_fret': None, 'method': 'old_music_theory_greedy'})
            pred_rows.append(row)
            continue

        best = max(
            positions,
            key=lambda p: old_position_score(n['midi'], p, note_row=n, previous_position=previous_position)
        )
        row.update({'pred_string': best['string'], 'pred_fret': best['fret'], 'method': 'old_music_theory_greedy'})
        pred_rows.append(row)
        previous_position = best

    return pred_rows


# -----------------------------------------------------------------------------
# Original/simple Viterbi without the new playability/context rules
# -----------------------------------------------------------------------------

def original_candidate_groups_for_notes(group_notes, max_candidates=MAX_GROUP_CANDIDATES):
    """Simple original-style candidate groups.

    This uses valid fretboard positions and a small low-fret preference, but does not use
    the new playability span penalties, awkward fingering penalties, open-string context,
    or chord/key context costs.
    """
    position_lists = []
    for n in group_notes:
        pos = get_possible_positions(n['midi'])
        if not pos:
            return []
        position_lists.append(pos)

    candidates = []
    for combo in product(*position_lists):
        combo = list(combo)

        # Keep physically impossible chord shapes out, but otherwise keep this simple.
        if len(combo) > 1 and len({p['string'] for p in combo}) != len(combo):
            continue

        frets = [p['fret'] for p in combo]
        base_cost = 0.05 * float(np.mean(frets)) + 0.05 * float(np.std(frets))
        candidates.append(enrich_candidate({'positions': combo, 'base_cost': base_cost}))

    return sorted(candidates, key=lambda c: c['base_cost'])[:max_candidates]


def original_transition_cost_matrix(prev_cands, curr_cands):
    """Movement-only transition cost for the simple/original Viterbi method."""
    prev_center = np.array([c['center'] for c in prev_cands], dtype=float)
    curr_center = np.array([c['center'] for c in curr_cands], dtype=float)
    prev_str = np.array([c['avg_string'] for c in prev_cands], dtype=float)
    curr_str = np.array([c['avg_string'] for c in curr_cands], dtype=float)

    mat = np.abs(prev_center[:, None] - curr_center[None, :])
    mat += 0.25 * np.abs(prev_str[:, None] - curr_str[None, :])
    return mat


def assign_viterbi_original(notes):
    """Simple/original Viterbi assignment for comparison with new playability Viterbi."""
    groups = group_notes_by_onset(notes)
    all_candidates = [original_candidate_groups_for_notes(g) for g in groups]
    if any(len(cands) == 0 for cands in all_candidates):
        raise ValueError('At least one group has no valid candidates.')

    dp = [np.array([c['base_cost'] for c in all_candidates[0]], dtype=float)]
    backptr = [np.full(len(dp[0]), -1, dtype=int)]

    for i in range(1, len(groups)):
        prev_cands, curr_cands = all_candidates[i - 1], all_candidates[i]
        trans = original_transition_cost_matrix(prev_cands, curr_cands)
        curr_base = np.array([c['base_cost'] for c in curr_cands], dtype=float)
        scores = dp[i - 1][:, None] + trans + curr_base[None, :]
        curr_back = np.argmin(scores, axis=0).astype(int)
        curr_costs = scores[curr_back, np.arange(scores.shape[1])]
        dp.append(curr_costs)
        backptr.append(curr_back)

    idx = int(np.argmin(dp[-1]))
    chosen_indices = [idx]
    for i in range(len(groups) - 1, 0, -1):
        idx = int(backptr[i][idx])
        chosen_indices.append(idx)
    chosen_indices = list(reversed(chosen_indices))

    pred_rows = []
    for g, cands, ci in zip(groups, all_candidates, chosen_indices):
        positions = cands[ci]['positions']
        for n, p in zip(g, positions):
            row = dict(n)
            row.update({'pred_string': p['string'], 'pred_fret': p['fret'], 'method': 'viterbi_original'})
            pred_rows.append(row)

    return sorted(pred_rows, key=lambda x: (x['start'], x.get('true_string') if x.get('true_string') is not None else 99, x['midi']))


# -----------------------------------------------------------------------------
# Combined-all method: original theory score + new playability rules + Viterbi
# -----------------------------------------------------------------------------

def old_theory_group_cost(group_notes, group_positions):
    """Convert the old higher-is-better music theory score into a lower-is-better cost."""
    if not group_notes or not group_positions:
        return 0.0
    scores = []
    for n, p in zip(group_notes, group_positions):
        scores.append(old_position_score(n['midi'], p, note_row=n, previous_position=None))
    # Negative because our Viterbi minimizes cost. Scale modestly so it helps but does not dominate playability.
    return -0.35 * float(np.mean(scores))


def candidate_groups_combined_all(group_notes, max_candidates=MAX_GROUP_CANDIDATES):
    """Candidate generator combining all available signals.

    Includes:
    - valid string/fret lookup
    - duplicate-string constraint for chords
    - new playability span/stretch/open-string rules
    - new key/chord context penalties
    - old teammate music-theory score as a bonus
    """
    position_lists = []
    for n in group_notes:
        pos = get_possible_positions(n['midi'])
        if not pos:
            return []
        position_lists.append(pos)

    candidates = []
    for combo in product(*position_lists):
        combo = list(combo)
        if len(combo) > 1 and len({p['string'] for p in combo}) != len(combo):
            continue

        base_cost = group_playability_cost(combo) + context_cost(group_notes, combo) + old_theory_group_cost(group_notes, combo)
        if math.isfinite(base_cost):
            candidates.append(enrich_candidate({'positions': combo, 'base_cost': base_cost}))

    if not candidates:
        return candidate_groups_for_notes(group_notes, max_candidates=max_candidates)

    return sorted(candidates, key=lambda c: c['base_cost'])[:max_candidates]


def assign_combined_all(notes):
    """Full combined method: original theory + new playability + Viterbi sequence optimization."""
    groups = group_notes_by_onset(notes)
    all_candidates = [candidate_groups_combined_all(g) for g in groups]
    if any(len(cands) == 0 for cands in all_candidates):
        raise ValueError('At least one group has no valid candidates.')

    dp = [np.array([c['base_cost'] for c in all_candidates[0]], dtype=float)]
    backptr = [np.full(len(dp[0]), -1, dtype=int)]

    for i in range(1, len(groups)):
        prev_cands, curr_cands = all_candidates[i - 1], all_candidates[i]
        trans = transition_cost_matrix(prev_cands, curr_cands)
        curr_base = np.array([c['base_cost'] for c in curr_cands], dtype=float)
        scores = dp[i - 1][:, None] + trans + curr_base[None, :]
        curr_back = np.argmin(scores, axis=0).astype(int)
        curr_costs = scores[curr_back, np.arange(scores.shape[1])]
        dp.append(curr_costs)
        backptr.append(curr_back)

    idx = int(np.argmin(dp[-1]))
    chosen_indices = [idx]
    for i in range(len(groups) - 1, 0, -1):
        idx = int(backptr[i][idx])
        chosen_indices.append(idx)
    chosen_indices = list(reversed(chosen_indices))

    pred_rows = []
    for g, cands, ci in zip(groups, all_candidates, chosen_indices):
        positions = cands[ci]['positions']
        for n, p in zip(g, positions):
            row = dict(n)
            row.update({'pred_string': p['string'], 'pred_fret': p['fret'], 'method': 'combined_all'})
            pred_rows.append(row)

    return sorted(pred_rows, key=lambda x: (x['start'], x.get('true_string') if x.get('true_string') is not None else 99, x['midi']))


# Quick smoke test on the first record.
smoke_notes = enrich_notes_with_context(records[0])[:50]
for name, fn in {
    'old_music_theory_greedy': assign_old_music_theory_greedy,
    'viterbi_original': assign_viterbi_original,
    'combined_all': assign_combined_all,
}.items():
    smoke_pred = fn(smoke_notes)
    print(f'{name}: produced {len(smoke_pred)} predictions')


old_music_theory_greedy: produced 50 predictions
viterbi_original: produced 50 predictions
combined_all: produced 50 predictions


## 11. Evaluation Metrics

In [14]:
def add_prediction_diagnostics(df):
    df = df.copy()
    df['pred_midi'] = [OPEN_STRING_MIDI[int(s)] + int(f) if pd.notna(s) and pd.notna(f) else np.nan for s, f in zip(df['pred_string'], df['pred_fret'])]
    df['correct_pitch_from_tab'] = df['pred_midi'] == df['midi']
    df['valid_position'] = df.apply(lambda r: pd.notna(r['pred_string']) and pd.notna(r['pred_fret']) and 0 <= int(r['pred_string']) <= 5 and 0 <= int(r['pred_fret']) <= MAX_FRET, axis=1)
    df['exact_position_correct'] = (df['pred_string'] == df['true_string']) & (df['pred_fret'] == df['true_fret'])
    df['string_correct'] = df['pred_string'] == df['true_string']
    df['fret_correct'] = df['pred_fret'] == df['true_fret']
    df['fret_error'] = (df['pred_fret'] - df['true_fret']).abs()
    df['string_error'] = (df['pred_string'] - df['true_string']).abs()
    return df

def duplicate_string_violation_rate(df, onset_tolerance=ONSET_TOLERANCE_SECONDS):
    if df.empty:
        return np.nan
    groups = group_notes_by_onset(df.sort_values(['start', 'midi']).to_dict('records'), tolerance=onset_tolerance)
    violations, total_chord_groups = 0, 0
    for g in groups:
        if len(g) <= 1:
            continue
        total_chord_groups += 1
        strings = [x.get('pred_string') for x in g if pd.notna(x.get('pred_string'))]
        if len(strings) != len(set(strings)):
            violations += 1
    return violations / total_chord_groups if total_chord_groups else 0.0

def average_group_span(df):
    if df.empty:
        return np.nan
    groups = group_notes_by_onset(df.sort_values(['start', 'midi']).to_dict('records'))
    spans = []
    for g in groups:
        frets = [int(x['pred_fret']) for x in g if pd.notna(x.get('pred_fret'))]
        if frets:
            spans.append(group_span(frets))
    return float(np.mean(spans)) if spans else np.nan

def movement_metrics(df):
    groups = group_notes_by_onset(df.sort_values(['start', 'midi']).to_dict('records'))
    centers, avg_strings = [], []
    for g in groups:
        frets = [int(x['pred_fret']) for x in g if pd.notna(x.get('pred_fret'))]
        strings = [int(x['pred_string']) for x in g if pd.notna(x.get('pred_string'))]
        if frets and strings:
            centers.append(estimate_hand_position_from_frets(frets))
            avg_strings.append(float(np.mean(strings)))
    if len(centers) <= 1:
        return {'avg_fret_jump': 0.0, 'avg_string_jump': 0.0, 'large_jump_rate': 0.0, 'large_jump_count': 0}
    fret_jumps = np.abs(np.diff(centers))
    string_jumps = np.abs(np.diff(avg_strings))
    large = fret_jumps > LARGE_JUMP_THRESHOLD
    return {'avg_fret_jump': float(np.mean(fret_jumps)), 'avg_string_jump': float(np.mean(string_jumps)), 'large_jump_rate': float(np.mean(large)), 'large_jump_count': int(np.sum(large))}

def evaluate_predictions(pred_rows):
    df = pd.DataFrame(pred_rows)
    if df.empty:
        return {}, df
    df = add_prediction_diagnostics(df)
    mv = movement_metrics(df)
    metrics = {
        'n_notes': len(df),
        'exact_position_acc': float(df['exact_position_correct'].mean()),
        'string_acc': float(df['string_correct'].mean()),
        'fret_acc': float(df['fret_correct'].mean()),
        'avg_fret_error': float(df['fret_error'].mean()),
        'avg_string_error': float(df['string_error'].mean()),
        'correct_pitch_from_tab_rate': float(df['correct_pitch_from_tab'].mean()),
        'valid_position_rate': float(df['valid_position'].mean()),
        'duplicate_string_violation_rate': float(duplicate_string_violation_rate(df)),
        'avg_group_span': float(average_group_span(df)),
        **mv,
    }
    return metrics, df


## 12. Run Full Evaluation

This evaluates **all available JAMS files** under `DATA_ROOT`. There is no note-count demo limit in this version.

Output is saved as CSV files in:

```text
/content/drive/MyDrive/Capstone/outputs/fretboard_playability/
```

In Google Drive, look under:

```text
My Drive / Capstone / outputs / fretboard_playability
```


In [15]:
ASSIGNMENT_METHODS = {
    # Original/simple methods
    'lowest_fret': assign_baseline_lowest_fret,
    'highest_string': assign_baseline_highest_string,
    'old_music_theory_greedy': assign_old_music_theory_greedy,
    'viterbi_original': assign_viterbi_original,

    # Existing/newer baselines and algorithms
    'nearest_previous': assign_nearest_previous,
    'viterbi_playability': assign_viterbi_playability,

    # Full combined method
    'combined_all': assign_combined_all,
}

all_metrics, all_predictions, failed = [], [], []
for record in records:
    print(f"Running {record['recording']}...")
    notes = enrich_notes_with_context(record)
    notes = [n for n in notes if n.get('true_string') is not None and n.get('true_fret') is not None and 0 <= n['true_fret'] <= MAX_FRET]
    if not notes:
        failed.append({'recording': record['recording'], 'error': 'no usable notes'})
        continue
    for method_name, assign_fn in ASSIGNMENT_METHODS.items():
        print(f"  - {method_name}")
        try:
            pred = assign_fn(notes)
            metrics, pred_df = evaluate_predictions(pred)
            metrics.update({
                'recording': record['recording'],
                'method': method_name,
                'is_solo': record['recording'].endswith('_solo'),
                'is_comp': record['recording'].endswith('_comp')
            })
            all_metrics.append(metrics)
            pred_df['recording'] = record['recording']
            pred_df['method'] = method_name
            all_predictions.append(pred_df)
        except Exception as e:
            failed.append({'recording': record['recording'], 'method': method_name, 'error': repr(e)})

metrics_df = pd.DataFrame(all_metrics)
predictions_df = pd.concat(all_predictions, ignore_index=True) if all_predictions else pd.DataFrame()
failed_df = pd.DataFrame(failed)

summary_cols = dict(
    recordings=('recording', 'nunique'),
    n_notes=('n_notes', 'sum'),
    exact_position_acc=('exact_position_acc', 'mean'),
    string_acc=('string_acc', 'mean'),
    fret_acc=('fret_acc', 'mean'),
    avg_fret_error=('avg_fret_error', 'mean'),
    avg_string_error=('avg_string_error', 'mean'),
    correct_pitch_from_tab_rate=('correct_pitch_from_tab_rate', 'mean'),
    valid_position_rate=('valid_position_rate', 'mean'),
    avg_fret_jump=('avg_fret_jump', 'mean'),
    avg_string_jump=('avg_string_jump', 'mean'),
    large_jump_rate=('large_jump_rate', 'mean'),
    duplicate_string_violation_rate=('duplicate_string_violation_rate', 'mean'),
    avg_group_span=('avg_group_span', 'mean'),
)

summary_df = metrics_df.groupby('method', as_index=False).agg(**summary_cols).sort_values(
    ['exact_position_acc', 'avg_fret_jump'], ascending=[False, True]
)
solo_summary_df = metrics_df[metrics_df['is_solo']].groupby('method', as_index=False).agg(**summary_cols).sort_values(
    ['exact_position_acc', 'avg_fret_jump'], ascending=[False, True]
)

# A compact class-facing comparison table.
comparison_cols = [
    'method',
    'recordings',
    'n_notes',
    'exact_position_acc',
    'string_acc',
    'fret_acc',
    'avg_fret_error',
    'avg_string_error',
    'avg_fret_jump',
    'large_jump_rate',
    'duplicate_string_violation_rate',
    'avg_group_span',
    'valid_position_rate',
    'correct_pitch_from_tab_rate',
]
comparison_table = summary_df[comparison_cols].copy()
solo_comparison_table = solo_summary_df[comparison_cols].copy()

# Round for readability without changing the saved raw metrics.
round_cols = [c for c in comparison_cols if c not in ['method', 'recordings', 'n_notes']]
comparison_table[round_cols] = comparison_table[round_cols].round(4)
solo_comparison_table[round_cols] = solo_comparison_table[round_cols].round(4)

metrics_path = OUTPUT_DIR / 'fretboard_eval_by_recording.csv'
summary_path = OUTPUT_DIR / 'fretboard_eval_summary_all.csv'
solo_summary_path = OUTPUT_DIR / 'fretboard_eval_summary_solo.csv'
comparison_path = OUTPUT_DIR / 'fretboard_method_comparison_table.csv'
solo_comparison_path = OUTPUT_DIR / 'fretboard_method_comparison_table_solo.csv'
pred_path = OUTPUT_DIR / 'fretboard_predictions_all.csv'
failed_path = OUTPUT_DIR / 'fretboard_failed_runs.csv'

metrics_df.to_csv(metrics_path, index=False)
summary_df.to_csv(summary_path, index=False)
solo_summary_df.to_csv(solo_summary_path, index=False)
comparison_table.to_csv(comparison_path, index=False)
solo_comparison_table.to_csv(solo_comparison_path, index=False)
predictions_df.to_csv(pred_path, index=False)
failed_df.to_csv(failed_path, index=False)

print('Saved outputs:')
for p in [metrics_path, summary_path, solo_summary_path, comparison_path, solo_comparison_path, pred_path, failed_path]:
    print(' -', p.resolve())

print('\nAll-recording method comparison table:')
display(comparison_table)

print('\nSolo-only method comparison table:')
display(solo_comparison_table)

print('\nFull all-recording summary:')
display(summary_df)

print('\nFull solo-only summary:')
display(solo_summary_df)

if not failed_df.empty:
    print('\nFailed runs:')
    display(failed_df)


Running 00_BN1-129-Eb_comp...
  - lowest_fret
  - highest_string
  - old_music_theory_greedy
  - viterbi_original
  - nearest_previous
  - viterbi_playability
  - combined_all
Running 00_BN1-129-Eb_solo...
  - lowest_fret
  - highest_string
  - old_music_theory_greedy
  - viterbi_original
  - nearest_previous
  - viterbi_playability
  - combined_all
Running 00_BN1-147-Gb_comp...
  - lowest_fret
  - highest_string
  - old_music_theory_greedy
  - viterbi_original
  - nearest_previous
  - viterbi_playability
  - combined_all
Running 00_BN1-147-Gb_solo...
  - lowest_fret
  - highest_string
  - old_music_theory_greedy
  - viterbi_original
  - nearest_previous
  - viterbi_playability
  - combined_all
Running 00_BN2-131-B_comp...
  - lowest_fret
  - highest_string
  - old_music_theory_greedy
  - viterbi_original
  - nearest_previous
  - viterbi_playability
  - combined_all
Running 00_BN2-131-B_solo...
  - lowest_fret
  - highest_string
  - old_music_theory_greedy
  - viterbi_original
  - near

,method,recordings,n_notes,exact_position_acc,string_acc,fret_acc,avg_fret_error,avg_string_error,avg_fret_jump,large_jump_rate,duplicate_string_violation_rate,avg_group_span,valid_position_rate,correct_pitch_from_tab_rate
0,combined_all,360,62476,0.5846,0.5846,0.5846,2.1051,0.4541,0.9249,0.0003,0.0000,0.5761,1.0,1.0
5,viterbi_original,360,62476,0.5667,0.5667,0.5667,2.1783,0.4710,0.9037,0.0002,0.0000,0.9172,1.0,1.0
6,viterbi_playability,360,62476,0.5618,0.5618,0.5618,2.3946,0.5130,0.9228,0.0004,0.0000,0.5916,1.0,1.0
4,old_music_theory_greedy,360,62476,0.4327,0.4327,0.4327,3.0425,0.6554,1.6571,0.0509,0.2398,0.5321,1.0,1.0
3,nearest_previous,360,62476,0.4321,0.4321,0.4321,4.2315,0.8867,1.0748,0.0054,0.0000,0.6513,1.0,1.0
1,highest_string,360,62476,0.3951,0.3951,0.3951,3.2917,0.7093,1.4329,0.0104,0.3354,0.6309,1.0,1.0
2,lowest_fret,360,62476,0.3951,0.3951,0.3951,3.2917,0.7093,1.4329,0.0104,0.3354,0.6309,1.0,1.0



Solo-only method comparison table:


,method,recordings,n_notes,exact_position_acc,string_acc,fret_acc,avg_fret_error,avg_string_error,avg_fret_jump,large_jump_rate,duplicate_string_violation_rate,avg_group_span,valid_position_rate,correct_pitch_from_tab_rate
0,combined_all,180,16861,0.4744,0.4744,0.4744,2.7265,0.5940,1.1885,0.0001,0.0000,0.1103,1.0,1.0
5,viterbi_original,180,16861,0.4730,0.4730,0.4730,2.6521,0.5805,1.2177,0.0001,0.0000,0.1370,1.0,1.0
6,viterbi_playability,180,16861,0.4262,0.4262,0.4262,3.2705,0.7048,1.1862,0.0001,0.0000,0.1110,1.0,1.0
4,old_music_theory_greedy,180,16861,0.3512,0.3512,0.3512,3.6010,0.7827,1.8418,0.0480,0.3050,0.0495,1.0,1.0
1,highest_string,180,16861,0.2991,0.2991,0.2991,3.9326,0.8528,1.7738,0.0135,0.3989,0.0964,1.0,1.0
2,lowest_fret,180,16861,0.2991,0.2991,0.2991,3.9326,0.8528,1.7738,0.0135,0.3989,0.0964,1.0,1.0
3,nearest_previous,180,16861,0.2551,0.2551,0.2551,6.1787,1.2925,1.3419,0.0035,0.0000,0.1156,1.0,1.0



Full all-recording summary:


,method,recordings,n_notes,exact_position_acc,string_acc,fret_acc,avg_fret_error,avg_string_error,correct_pitch_from_tab_rate,valid_position_rate,avg_fret_jump,avg_string_jump,large_jump_rate,duplicate_string_violation_rate,avg_group_span
0,combined_all,360,62476,0.584554,0.584554,0.584554,2.105127,0.454092,1.0,1.0,0.924935,0.776824,0.000311,0.000000,0.576083
5,viterbi_original,360,62476,0.566676,0.566676,0.566676,2.178295,0.470972,1.0,1.0,0.903708,0.774265,0.000198,0.000000,0.917174
6,viterbi_playability,360,62476,0.561757,0.561757,0.561757,2.394617,0.513044,1.0,1.0,0.922780,0.774790,0.000406,0.000000,0.591564
4,old_music_theory_greedy,360,62476,0.432711,0.432711,0.432711,3.042466,0.655416,1.0,1.0,1.657083,0.827389,0.050894,0.239822,0.532090
3,nearest_previous,360,62476,0.432134,0.432134,0.432134,4.231474,0.886742,1.0,1.0,1.074808,0.752081,0.005384,0.000000,0.651281
1,highest_string,360,62476,0.395114,0.395114,0.395114,3.291667,0.709336,1.0,1.0,1.432889,0.757274,0.010401,0.335430,0.630912
2,lowest_fret,360,62476,0.395114,0.395114,0.395114,3.291667,0.709336,1.0,1.0,1.432889,0.757274,0.010401,0.335430,0.630912



Full solo-only summary:


,method,recordings,n_notes,exact_position_acc,string_acc,fret_acc,avg_fret_error,avg_string_error,correct_pitch_from_tab_rate,valid_position_rate,avg_fret_jump,avg_string_jump,large_jump_rate,duplicate_string_violation_rate,avg_group_span
0,combined_all,180,16861,0.474438,0.474438,0.474438,2.726468,0.594013,1.0,1.0,1.188477,0.419588,0.000090,0.000000,0.110262
5,viterbi_original,180,16861,0.472960,0.472960,0.472960,2.652101,0.580544,1.0,1.0,1.217713,0.441550,0.000116,0.000000,0.137039
6,viterbi_playability,180,16861,0.426173,0.426173,0.426173,3.270549,0.704824,1.0,1.0,1.186189,0.418411,0.000090,0.000000,0.111044
4,old_music_theory_greedy,180,16861,0.351216,0.351216,0.351216,3.600990,0.782748,1.0,1.0,1.841825,0.491197,0.048049,0.304985,0.049503
1,highest_string,180,16861,0.299113,0.299113,0.299113,3.932610,0.852759,1.0,1.0,1.773757,0.375817,0.013468,0.398890,0.096357
2,lowest_fret,180,16861,0.299113,0.299113,0.299113,3.932610,0.852759,1.0,1.0,1.773757,0.375817,0.013468,0.398890,0.096357
3,nearest_previous,180,16861,0.255143,0.255143,0.255143,6.178749,1.292517,1.0,1.0,1.341906,0.393128,0.003451,0.000000,0.115636


## 13. Per-Recording Metrics

In [16]:
display(metrics_df.sort_values(['recording', 'method']).reset_index(drop=True))


,n_notes,exact_position_acc,string_acc,fret_acc,avg_fret_error,avg_string_error,correct_pitch_from_tab_rate,valid_position_rate,duplicate_string_violation_rate,avg_group_span,avg_fret_jump,avg_string_jump,large_jump_rate,large_jump_count,recording,method,is_solo,is_comp
0,133,0.857143,0.857143,0.857143,0.684211,0.142857,1.0,1.0,0.000000,0.828947,0.920000,1.017778,0.000000,0,00_BN1-129-Eb_comp,combined_all,False,True
1,133,0.105263,0.105263,0.105263,4.203008,0.894737,1.0,1.0,0.250000,1.026316,1.093333,1.002222,0.000000,0,00_BN1-129-Eb_comp,highest_string,False,True
2,133,0.105263,0.105263,0.105263,4.203008,0.894737,1.0,1.0,0.250000,1.026316,1.093333,1.002222,0.000000,0,00_BN1-129-Eb_comp,lowest_fret,False,True
3,133,0.609023,0.609023,0.609023,1.969925,0.406015,1.0,1.0,0.000000,1.065789,0.986667,1.042222,0.013333,1,00_BN1-129-Eb_comp,nearest_previous,False,True
4,133,0.270677,0.270677,0.270677,3.451128,0.729323,1.0,1.0,0.214286,0.842105,1.600000,1.080000,0.026667,2,00_BN1-129-Eb_comp,old_music_theory_greedy,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2515,93,0.182796,0.182796,0.182796,6.118280,1.322581,1.0,1.0,0.000000,0.000000,2.184783,0.130435,0.076087,7,05_SS3-98-C_solo,lowest_fret,True,False
2516,93,0.118280,0.118280,0.118280,9.053763,1.881720,1.0,1.0,0.000000,0.000000,1.619565,0.315217,0.000000,0,05_SS3-98-C_solo,nearest_previous,True,False
2517,93,0.118280,0.118280,0.118280,6.333333,1.376344,1.0,1.0,0.000000,0.000000,2.565217,0.336957,0.097826,9,05_SS3-98-C_solo,old_music_theory_greedy,True,False
2518,93,0.430108,0.430108,0.430108,3.376344,0.741935,1.0,1.0,0.000000,0.000000,1.423913,0.347826,0.000000,0,05_SS3-98-C_solo,viterbi_original,True,False


## 14. Error Analysis Examples

This table shows examples where the predicted tab is pitch-valid but does not match GuitarSet's exact string/fret annotation. These are useful for explaining why exact tab recovery is stricter than playable tab generation.

In [17]:
if not predictions_df.empty:
    diag = add_prediction_diagnostics(predictions_df)
    error_examples = (
        diag[(diag['method'] == 'viterbi_playability') & (~diag['exact_position_correct']) & (diag['correct_pitch_from_tab'])]
        .sort_values(['recording', 'start'])
        [['recording', 'start', 'midi', 'chord_label', 'key_label', 'true_string', 'true_fret', 'pred_string', 'pred_fret', 'fret_error', 'string_error', 'correct_pitch_from_tab']]
        .head(50)
    )
    error_path = OUTPUT_DIR / 'fretboard_error_examples.csv'
    error_examples.to_csv(error_path, index=False)
    print('Saved error examples:', error_path.resolve())
    display(error_examples)


Saved error examples: /content/drive/.shortcut-targets-by-id/1JNqe8bukG93wCWVxbk7SKlNvVyIZHyTC/Capstone/outputs/fretboard_playability/fretboard_error_examples.csv


,recording,start,midi,chord_label,key_label,true_string,true_fret,pred_string,pred_fret,fret_error,string_error,correct_pitch_from_tab
674,00_BN1-129-Eb_comp,1.887252,51,D#:maj,Eb:major,1,6,0,11,5,1,True
688,00_BN1-129-Eb_comp,3.753533,65,D#:maj,Eb:major,4,6,3,10,4,1,True
694,00_BN1-129-Eb_comp,4.675528,65,D#:maj,Eb:major,4,6,3,10,4,1,True
699,00_BN1-129-Eb_comp,5.355256,51,D#:maj,Eb:major,1,6,0,11,5,1,True
744,00_BN1-129-Eb_comp,12.805098,51,D#:maj,Eb:major,1,6,0,11,5,1,True
745,00_BN1-129-Eb_comp,13.304395,51,D#:maj,Eb:major,1,6,0,11,5,1,True
761,00_BN1-129-Eb_comp,16.512240,58,A#:maj,Eb:major,2,8,3,3,5,1,True
763,00_BN1-129-Eb_comp,17.020971,65,G#:maj,Eb:major,4,6,5,1,5,1,True
776,00_BN1-129-Eb_comp,19.085891,51,D#:maj,Eb:major,1,6,0,11,5,1,True
1290,00_BN1-129-Eb_solo,0.187705,58,D#:maj,Eb:major,2,8,3,3,5,1,True


In [18]:
print('Final output folder:')
print(OUTPUT_DIR.resolve())
print('\nCSV files currently in output folder:')
for f in sorted(OUTPUT_DIR.glob('*.csv')):
    print(' -', f.name)


Final output folder:
/content/drive/.shortcut-targets-by-id/1JNqe8bukG93wCWVxbk7SKlNvVyIZHyTC/Capstone/outputs/fretboard_playability

CSV files currently in output folder:
 - fretboard_error_examples.csv
 - fretboard_eval_by_recording.csv
 - fretboard_eval_summary_all.csv
 - fretboard_eval_summary_solo.csv
 - fretboard_failed_runs.csv
 - fretboard_method_comparison_table.csv
 - fretboard_method_comparison_table_solo.csv
 - fretboard_predictions_all.csv


## 15. Quick Interpretation Template

Use this language in class when presenting the output:

> We evaluated fretboard assignment separately from note transcription by using GuitarSet ground-truth MIDI notes as input. This isolates the tab-assignment problem. We compare simple baselines against a Viterbi-style playability algorithm. Exact string/fret accuracy measures how often we recover the original annotated fingering, while validity and playability metrics measure whether the generated tab is mechanically correct and comfortable. Because the same pitch can be played in multiple valid guitar positions, exact recovery is useful but not the only success metric.
